# Vision Transformer (ViT)

La seguente figura mostra una panoramica dell'architettura Vision Transformer:

<div style="background-color: white; display: inline-block; padding: 10px; width:1000px">
    <img src="img/architecture.png" />
</div>

I principali ingredienti da considerare sono i seguenti:
1. Embedding Layer: tokenizzazione dell'immagine e aggiunta di un token "speciale" (CLS Token)
2. Funzione di Scaled Dot Product Attention
3. Attention Head Layer
4. Multihead Attention Layer
5. FeedForward Layer
6. Transformer Encoder Layer

Li implementeremo uno a uno utilizzando il framework **PyTorch**

Le classi in questo framework ereditano dalla class base "torch.nn.Module" e hanno bisogno di: <br/>
(1) un metodo costuttore base (i.e. __init__() ) che indica i vari "componenti" dell'architettura <br/>
(2) un metodo che indichi come la classe deve elaborare i dati di ingresso nei dati di uscita (i.e. forward() )

In [1]:
import torch

## 1. Embedding Layer: tokenizzazione dell'immagine e aggiunta di un token "speciale" (CLS Token)

Il punto di partenza dell'architettura Vision Transformer (ViT) è l'utilizzo dei token come descrittori di un'immagine. I token si possono intendere come le unità fondamentali di linguaggio tramite cui il sistema interpreta l'immagine.

Nello specifico vorremmo:<br/>
a. Estrarre delle piccole patch (i.e. sotto-porzioni di immagine) a partire dall'immagine originale<br/>
b. Codificare il contenuto di ciascuna patch in un vettore addestrabile di dimensione configurabile (i.e. che verrà tarato in automatico sulla base dei dati forniti)<br/>
c. Aggiungere l'informazione della posizione spaziale per rendere la rete neurale "consapevole" della posizione e della relazione spaziale tra le varie patch estratte<br/>


La pipeline per passare dalle patch ai token dell'immagine è mostrata nella figura sottostante:

<br/>

<div style="background-color: white; display: inline-block; padding: 10px; width:700px">
    <img src="img/tokenization.png" />
</div>

Per ottenere dei vettori 1D a partire da immagini 3D (ossia 2D spaziali + 3 canali RGB), si applica una trasformazione lineare tramite una Projection Matrix.

Questa operazione viene implementata con dei filtri convoluzionali con kernel size pari allo stride, che svolgono automaticamente due compiti: (1) estraggono patch non sovrapposte dall'immagine originale, e (2) le proiettano in uno spazio vettoriale 1D (i.e. calcolano la projection matrix).

In [2]:
class EmbeddingLayer(torch.nn.Module):
    
    def __init__(self, imgSize: int, patchSize : int, embeddingDimension : int):

        # Inizializziamo la parent class
        super().__init__()

        # =========================================
        # Creiamo i vari "componenti" necessari
        # =========================================

        # 1. PROJECTION MATRIX

        # Projection Matrix (3 è il numero di canali in ingresso nell'immagine: RGB)
        self.projectionMatrix = torch.nn.Conv2d(in_channels = 3, 
                                                out_channels = embeddingDimension, 
                                                kernel_size = patchSize,
                                                stride = patchSize,
                                                bias = True)
        
        # Inizializziamo la Projection Matrix con numeri casuali secondo una Gaussiana troncata
        torch.nn.init.trunc_normal_(self.projectionMatrix.weight, std=0.02)

        # 2. CLS TOKEN
        
        # Aggiungiamo un token "speciale" che non è associato a nessuna porzione dell'immagine: è quindi un token allenabile, globale, che andrà a svolgere il ruolo di riassumere il contenuto complessivo preente nell'immagine
        # Lo inizializziamo con il valore zero
        self.clsToken = torch.nn.Parameter(torch.zeros(1, 1, embeddingDimension))

        # 3. POSITIONAL EMBEDDING

        # Calcoliamo il numero di patch totali (i.e. moltiplicazione di orizzontali e verticali) in cui viene scomposta l'immagine, assumendo che la forma di entrambe (i.e. immagine e patch) sia quadrata
        numPatches = (imgSize // patchSize) ** 2

        # Positional Embeddings (sono dei parameteri aggiuntivi addestrabili dal modello sulla base dei dati) -> Ne abbiamo uno per ciascuna patch estratta + 1 per il CLS token (in questo modo la rete può distinguere l'ordine tra le varie patch)
        self.positionalEmbedding = torch.nn.Parameter(torch.zeros(1, numPatches+1, embeddingDimension))

        # Inizializziamo i Positional Embeddings con numeri casuali secondo una Gaussiana troncata
        torch.nn.init.trunc_normal_(self.positionalEmbedding, std=0.02)

        



    def forward(self, x : torch.Tensor) -> torch.Tensor:

        # Estraiamo la dimensione del "batch" (i.e. le immagini sono elaborate in blocco e non una alla volta)
        B = x.shape[0]

        # =========================================
        # Applichiamo i "componenti" creati
        # =========================================

        # 1. PROJECTION MATRIX

        # Estraiamo dall'immagine le patch e le rappresentiamo sotto forma di vettori allenabili di dimensione "embeddingDimension"
        x = self.projectionMatrix(x) # [B, embeddingDimension, numPatches orizzontale, numPatches verticale]

        # Facciamo un flatten: non ci interessa preservare la struttura 2D che distingue tra patch lungo l'orizzontale e lungo il verticale
        x = x.flatten(start_dim=2) # [B, embeddingDimension, numPatches=(numPatches orizzontale*numPatches verticale)]

        # Invertiamo l'ordine delle dimensioni (1<->2) per avere l'embedding dimension come ultima dimensione del tensore (è solo per preservare la compatibilità)
        x = x.transpose(1,2) # [B,  numPatches, embeddingDimension]

        # 2. CLS TOKEN

        # Espandiamo il CLS token così da averne uno per ogni immagine nel batch -> Non fa la copia, ma modifica la view; -1 significa "non modificare lungo quella dimensione"
        cls = self.clsToken.expand(B, -1, -1) # [B, 1, embeddingDimension]

        # Concateniamo il CLS token
        x = torch.cat([cls, x], dim=1) # [B,  numPatches+1, embeddingDimension]

        # 3. POSITIONAL EMBEDDING

        x = x + self.positionalEmbedding # [B,  numPatches+1, embeddingDimension]

        return x

## 2. Funzione di Scaled Dot Product Attention

La Scaled Dot Product Attention si basa su 3 vettori fondamentali:
- Key (K)
- Query (Q)
- Value (V)

Il funzionamento è simile a quello di una look-up table: si fa una domanda (query) e si ricerca nel dizionario se esiste una chiave (key) che contiene la riposta alla nostra domanda. Se si trova il risultato si estrae il valore (Value) indicato dalla chiave. 

La funzione di Scaled Dot Product Attention estende questo concetto base permettendo di ottenere non solo le chiavi che matchano perfettamente con la nostra domanda, ma anche chiavi (keys) relativamente simili a quello che stiamo chiedendo (query). Si passa quindi da un sistema hard-retrival ad uno soft-retrival.

Nello specifico, per misurare la soglianza tra Query (Q) e Key (k), si calcola il prodotto scalare tra questi due vettori e si ottiene uno scalare che quantifica quanto questi vettori sono allineati tra loro. L'ipotesi di base è che vettori allineati indicano concetti e informazioni simili tra loro, mentre vettori disallineati indicano dissomiglianza tra la domanda (Q) che stiamo facendo noi e il template della domanda presente nel sistema (ossia la Key, K).

Un esempio applicativo per rendere l'idea viene mostrato nell'immagine seguente:

<br/>

<div style="background-color: white; display: inline-block; padding: 10px; width:700px">
    <img src="img/scaled_dot_product.png" />
</div>

In [3]:
def scaledDotProductAttention(query, key, value):

    # La dimensione di embedding è l'ultima (vedi sopra)
    embeddingDim = query.shape[-1]

    # Facciamo una Batch Matrix Multiplication (i.e. moltiplicazione tra matrici orttimizzata per batch) X dot Y = XY^T -> Diventano matrici aggiungendo la dimensione del batch
    scores = torch.bmm(query, key.transpose(1,2)) / embeddingDim ** 0.5 # Divide for square root of embedding dimension due to variance

    # Normalize between 0 and 1 so to have a score for each element along the embedding dimension
    attentionWeights = torch.softmax(scores, dim=-1) 

    # Extract the values by weighting them individually with the computed attention weights
    extractedValues = torch.bmm(attentionWeights, value)

    return extractedValues

## 3. Attention Head Layer

Un layer di attenzione (detto attention head) è uno strato di una rete neurale che implementa la funzione di "scaled dot product attention". 

Creiamo una classe in modo tale che i parametri della rete vengono salvati ngli attributi della classe stessa e possiamo riutilizzare quindi la funzione creata sopra.

Le matrici proiettano i dati dalla dimensione di embedding, definita prima, ad un'altra dimensione teoricamente liberamente scelta da noi e detta "head dimension", in pratica fissata ad un valore specifico nella classe successiva

In [4]:
class AttentionHead(torch.nn.Module):

    def __init__(self, embeddingDimension, headDimension):

        # Inizializzazione della classe parent
        super().__init__()

        # =========================================
        # Creiamo i vari "componenti" necessari
        # =========================================

        # Creiamo le matrici di query, key e value che prenderanno poi in ingresso il nostro dato "x" => Implementiamo il prodotto matriciale con linear, con bias = False (ossia y = Ax + b, con b=0)
        self.Q = torch.nn.Linear(in_features = embeddingDimension, 
                                 out_features = headDimension,
                                 bias = False)
        self.K = torch.nn.Linear(in_features = embeddingDimension, 
                                 out_features = headDimension,
                                 bias = False)
        self.V = torch.nn.Linear(in_features = embeddingDimension, 
                                 out_features = headDimension,
                                 bias = False)



    def forward(self, x):

        # =========================================
        # Applichiamo i "componenti" creati
        # =========================================

        # Facciamo passare x dentro alle matrici di query, key e value e calcoliamo l'output
        output = scaledDotProductAttention(query = self.Q(x), 
                                           key = self.K(x), 
                                           value = self.V(x))
        
        return output


## 4. Multihead Attention Layer

L'idea è quella di applicare multiple attention head in parallelo, ciascuna con le proprie matrici di key, query e value. 

In [5]:
class MultiHeadAttention(torch.nn.Module):

    def __init__(self, embeddingDimension, numHeads):

        # Inizializzazione della classe parent
        super().__init__()

        # =========================================
        # Creiamo i vari "componenti" necessari
        # =========================================

        # Come dimensione di ciascuna head utilizziamo la dimensione di embedding partizionata lungo le varie attention head disponibili (in questo modo è semplice mettere tante MultiHeadAttention in cascata)
        headDimension = embeddingDimension // numHeads

        # Creiamo una lista di attention heads
        self.attHeads = torch.nn.ModuleList([AttentionHead(embeddingDimension = embeddingDimension, headDimension = headDimension) for i in range(numHeads)])

        # Utilizziamo una matrice alla fine per mescolare l'output delle singole attention head
        self.linear = torch.nn.Linear(embeddingDimension, embeddingDimension)


    def forward(self, x):

        # =========================================
        # Applichiamo i "componenti" creati
        # =========================================

        # Passiamo i dati "x" dentro a ciascuna attention head e poi concateniamo il risultato. Per il modo in cui abbiamo scelto la headDimension, dopo la concatenazione otteniamo output che è lungo come embedingDimension
        x = torch.cat([head(x) for head in self.attHeads], dim=-1) # Concateniamo lungo l'ultima dimensione, ossia quella delle headDimension

        x = self.linear(x)

        return x

## 5. Feedforward Layer

Il layer di feedforward è una semplice rete densa del tipo MultiLayer Perceptron (a due strati) con funzione di attivazione GeLU e dropout per regolarizzare. Questo layer verrà infraposto tra la catena di MultiHeadAttention

In [6]:
class FeedForward(torch.nn.Module):

    def __init__(self, hiddenSize, intermediateSize, hiddenDropoutProb):

        # Inizializzazione della classe parent
        super().__init__()

        # =========================================
        # Creiamo i vari "componenti" necessari
        # =========================================

        self.linear1 = torch.nn.Linear(in_features = hiddenSize, 
                                       out_features = intermediateSize, 
                                       bias = True)
        self.activation = torch.nn.GELU()
        self.linear2 = torch.nn.Linear(in_features = intermediateSize, 
                                       out_features = hiddenSize, 
                                       bias = True)
        self.dropout = torch.nn.Dropout(p = hiddenDropoutProb)

    def forward(self, x):

        # =========================================
        # Applichiamo i "componenti" creati
        # =========================================

        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.dropout(x)
        
        return x

## 6. Transformer Encoder

Ora possiamo mettere insieme tutti i blocchi creati fin'ora, creando così una macrostruttura, riportata nell'immagine sotto, che prende il nome di Transformer Encoder. Questa struttura è la base dell'architettura Vision Transformer.

In questa struttura si notano due ulteriori layer: Layer normalization. Questi Layer sono già presenti nel framework di PyTorch e servono solo a riscalare l'output prodotto dai vari layer che abbiamo creato fin'ora, normalizzandolo per evitare che i valori processati diventino gradualmente troppo grossi o troppo piccoli.

<br/>

<div style="background-color: white; display: inline-block; padding: 10px; width:250px">
    <img src="img/encoderBlock.png" />
</div>


In [7]:
class TransformerEncoder(torch.nn.Module):

    def __init__(self, numAttentionHeads, hiddenSize, hiddenDropoutProb):

        # Inizializzazione della classe parent
        super().__init__()

        # =========================================
        # Creiamo i vari "componenti" necessari
        # =========================================

        self.layerNorm1 = torch.nn.LayerNorm(normalized_shape = hiddenSize)
        self.multiHeadAttention = MultiHeadAttention(hiddenSize, numAttentionHeads)
        self.layerNorm2 = torch.nn.LayerNorm(normalized_shape = hiddenSize)
        self.feedForward = FeedForward(hiddenSize = hiddenSize, 
                                       intermediateSize = hiddenSize, 
                                       hiddenDropoutProb = hiddenDropoutProb)



    def forward(self, x):

        # =========================================
        # Applichiamo i "componenti" creati
        # =========================================

        x = x + self.multiHeadAttention(self.layerNorm1(x))
        x = x + self.feedForward(self.layerNorm2(x))

        return x

## ViT

Possiamo ora mettere tutti i blocchi insieme e creare finalmente un Vision Transformer

In [8]:
class ViT(torch.nn.Module):

    def __init__(self, imgSize, patchSize, embeddingDimension, depthEncoder, numAttentionHeads, hiddenSize, hiddenDropoutProbab, numClasses):

        # Inizializzazione della classe parent
        super().__init__()

        # =========================================
        # Creiamo i vari "componenti" necessari
        # =========================================

        # Primo layer di embedding 
        self.embeddingLayer = EmbeddingLayer(imgSize, patchSize, embeddingDimension)

        # N layer di Transformer Encoder
        self.encoderLayers = torch.nn.ModuleList([TransformerEncoder(numAttentionHeads=numAttentionHeads, 
                                                                     hiddenSize=hiddenSize, 
                                                                     hiddenDropoutProb=hiddenDropoutProbab) for i in range(depthEncoder)])
        
        # Layer Finale denso (i.e. MLP) per la classificazione finale
        self.linear = torch.nn.Linear(in_features = hiddenSize,
                                      out_features = numClasses)
        

    def forward(self, x):

        # =========================================
        # Applichiamo i "componenti" creati
        # =========================================

        # Applichiamo il layer di embedding
        x = self.embeddingLayer(x)

        # Applichiamo la cascata di layer di encoder
        for layer in self.encoderLayers:
            x = layer(x)

        # Estraiamo il token CLS e usiamo quello per la classificazione finale
        x = self.linear(x[:, 0])

        return x

# Allenamento di esempio

In [9]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [10]:
# Iperparametri
IMG_SIZE          = 32 
PATCH_SIZE        = 4 
EMBEDDING_DIM     = 64
HIDDEN_SIZE       = 64
NUM_HEADS         = 4    
DEPTH_ENCODER     = 3    
HIDDEN_DROPOUT    = 0.1
NUM_CLASSES       = 10   

BATCH_SIZE        = 32
NUM_EPOCHS        = 20
LEARNING_RATE     = 1e-3
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
# Definizione dataset e data augmentation
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_transform)
val_dataset   = datasets.CIFAR10(root="./data", train=False, download=True, transform=val_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Modello
model = ViT(imgSize = IMG_SIZE,
            patchSize = PATCH_SIZE,
            embeddingDimension = EMBEDDING_DIM,
            depthEncoder = DEPTH_ENCODER,
            numAttentionHeads = NUM_HEADS,
            hiddenSize = HIDDEN_SIZE,
            hiddenDropoutProbab = HIDDEN_DROPOUT,
            numClasses = NUM_CLASSES
        ).to(DEVICE)

In [13]:
# CrossEntropyLoss include già la Softmax internamente
criterion = torch.nn.CrossEntropyLoss()

# AdamW + cosine LR schedule
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [14]:
# Funzione helper di allenamento
def trainOneEpoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


# Funzione helper di valutazione
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total

In [15]:
# Formattazione a tabella
print(f"Training on {DEVICE} — {sum(p.numel() for p in model.parameters()):,} parametri\n")
print(f"{'Epoch':>6}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}")
print("─" * 55)

# Ciclo di allenamento
best_val_acc = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = trainOneEpoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)
    scheduler.step()

    print(f"{epoch:>6}  {train_loss:>10.4f}  {train_acc:>8.2%}  {val_loss:>8.4f}  {val_acc:>6.2%}")

    # Salva il checkpoint migliore
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_vit_cifar10.pth")

print(f"\nMigliore Val Accuracy: {best_val_acc:.2%}")
print("Checkpoint salvato in best_vit_cifar10.pth")

Training on cpu — 83,082 parametri

 Epoch  Train Loss  Train Acc  Val Loss  Val Acc
───────────────────────────────────────────────────────
     1      1.7849    33.94%    1.5056  44.56%
     2      1.5222    44.37%    1.4376  46.92%
     3      1.4448    47.41%    1.3339  51.59%
     4      1.3881    49.44%    1.3078  52.06%
     5      1.3422    51.14%    1.3364  52.15%
     6      1.3089    52.65%    1.2527  54.75%
     7      1.2759    53.62%    1.2598  54.44%
     8      1.2427    55.04%    1.1917  57.57%
     9      1.2132    56.23%    1.1722  57.85%
    10      1.1913    57.01%    1.1580  58.17%
    11      1.1621    58.16%    1.1674  57.93%
    12      1.1342    59.07%    1.1343  59.22%
    13      1.1157    59.92%    1.1066  60.23%
    14      1.0967    60.56%    1.0931  60.53%
    15      1.0755    61.22%    1.0872  60.97%
    16      1.0575    62.05%    1.0845  60.85%
    17      1.0399    62.66%    1.0752  61.49%
    18      1.0277    63.08%    1.0677  61.69%
    19      1